In [ ]:
!pip install nltk -q

In [ ]:
import re
import random
import unicodedata
import pandas as pd
from nltk.tokenize import RegexpTokenizer
from nltk.stem import SnowballStemmer

#Configuracion Inicial de Herramientas NLP


*   RegexpTokenizer (r'\w+) separa el texto en palabras.
*   SnowballStemmer('spanish') reduce cada palabra a su raiz (precio, precios -> preci)




In [51]:
from nltk import text
tokenizer = RegexpTokenizer(r'\w+')
stemmer = SnowballStemmer('spanish')

def normalizar_texto(texto):
  """
  Convierte texto a minisculas, elimina acentos y caracteres especiales.
  """
  texto = texto.lower()
  texto = unicodedata.normalize('NFKD', texto) # separa cada letra de su acento
  texto = texto.encode('ascii', 'ignore').decode('utf-8') # elimina acentos sueltos y simbolos
  texto = re.sub(r"[^a-zñ0-9\s]", ' ', texto) #pone en un espacio todo lo que no sea letra
  texto = re.sub(r"\s+"," ", texto).strip() #junta los espacios repetdisos en uno solo
  return texto

def tokenizar_y_stem(texto):
  """
  Tokeniza el texto y aplica stemming para reducir variaciones morfologicas
  """
  texto = normalizar_texto(texto)
  tokens = tokenizer.tokenize(texto)
  stem = [stemmer.stem(token) for token in tokens]
  return tokens, stem

In [52]:
print(normalizar_texto("¿Cuál es el PRECIO del Wi-Fi"))
print(tokenizar_y_stem("¿Cuanto cuestan los servicios?"))

cual es el precio del wi fi
(['cuanto', 'cuestan', 'los', 'servicios'], ['cuant', 'cuest', 'los', 'servici'])


#Definir intenciones del chatbot


Cada intencion tiene ejemplos (frases de entrenamiento) y respuestas (platnillas). Las plantillas pueden incluior {servicio}, que se rellena con la entidad detectada.


In [53]:
intenciones = {
    "saludo": {
        "ejemplos": ["hola", "buenos dias", "buenas tardes", "buenas noches","que tal", "necesito ayuda"],
        "respuestas": ["Hola, soy tu asistente virtual. Puedo ayudarte con horarios, precios, soporte y servicios.",
                       "Hola, con gusto te apoyo. Puedes preguntarme por horarios, precios o soporte tecnico."]
    },
    "consultar_horario": {
        "ejemplos": [
            "cual es su horario",
            "a que hora abren",
            "a que hora cierran",
            "horario de atencion",
            "trabajan los sabados",
            "abren los domingos"
         ],
        "respuestas": ["Nuestro horario de atencion es de lunes a viernes de 9:00 a 18:00 horas.",
                      "Atendemos de lunes a viernes de 9:00 a.m a 6:00 p.m. Los sabados atendemos con cita previa."]
    },
    "consultar_precio": {
        "ejemplos": ["cuanto cuesta el servicio","quiero saber el precio","cual es el costo", "me puedes dar la tarifa","precio de internet","precio de mantenimiento"],
        "respuestas": ["Los costos dependen del servicio solicitado. Detecte que te interesa: {servicio}. ¿Deseas una cotizacion general?",
                       "Puedo orientarte con los precios. El servicio identificado fue: {servicio}. ¿Deseas una cotizacion general?"]
    },
    "solicitar_soporte": {
        "ejemplos": ["tengo un problema", "no funciona mi cuenta", "necesito soporte","quiero levantar un reporte","fallo el internet","problema con mi servicio"],
        "respuestas":["Lamento el inconveniente. Detecte el servicio: {servicio}. Por favor describe el problema.",
                      "Con gusto te apoyo. Para iniciar el reporte, necesito que me indiques mas detalles sobre el problema en {servicio}."]
    },
    "despedida": {
        "ejemplos": ["gracias", "hasta luego", "adios", "nos vemos","eso es todo"],
        "respuestas": ["De nada. Que tengas un buen dia.", "Que tengas un buen dia.","Con gusto. Estoy disponible cuando necesites mas ayuda."]
    }
}

#Preparacion de vocabulario por intencion

Se unen los stems de todos los ejemplos de cada intencion en un conjunto.



In [54]:
def construir_vocabulario_intenciones(intenciones):
  """
  Construye una representacion basada en stems para cada intencion.
  """
  vocabulario = {}

  for nombre_intencion, datos in intenciones.items():
    stems_intencion = []

    for ejemplo in datos['ejemplos']:
      _, stems = tokenizar_y_stem(ejemplo)
      stems_intencion.extend(stems)

      vocabulario[nombre_intencion] = set(stems_intencion)
  return vocabulario

vocabulario_intenciones = construir_vocabulario_intenciones(intenciones)

for k, v in vocabulario_intenciones.items():
  print(f'{k:18s} -> {sorted(v)}')

saludo             -> ['ayud', 'buen', 'dias', 'hol', 'necesit', 'noch', 'que', 'tal', 'tard']
consultar_horario  -> ['a', 'abren', 'atencion', 'cierr', 'cual', 'de', 'doming', 'es', 'hor', 'horari', 'los', 'que', 'sab', 'su', 'trabaj']
consultar_precio   -> ['cost', 'cual', 'cuant', 'cuest', 'dar', 'de', 'el', 'es', 'internet', 'la', 'manten', 'me', 'preci', 'pued', 'quier', 'sab', 'servici', 'tarif']
solicitar_soporte  -> ['con', 'cuent', 'el', 'fall', 'funcion', 'internet', 'levant', 'mi', 'necesit', 'no', 'problem', 'quier', 'report', 'servici', 'soport', 'teng', 'un']
despedida          -> ['adi', 'es', 'eso', 'graci', 'hast', 'lueg', 'nos', 'tod', 'vem']


#Dteccion de Intencion

El puntaje de cada intencion es el numero de stems compartidos con el mensaje. Gana la intencion con mayor puntaje; si es 0, la intencion es desconocida (fallback).


In [55]:
def detectar_intencion(mensaje):
  """
  Detecta la intencion del mensaje. Compara stems del mensaje con el vocabulario
  de cada intencion. Devuelve intencion y puntuacion.
  """
  _, stems_mensaje = tokenizar_y_stem(mensaje)
  stems_mensaje = set(stems_mensaje)

  puntajes = {}

  for intencion, stems_intencion in vocabulario_intenciones.items():
    coincidencias = stems_mensaje.intersection(stems_intencion)
    puntajes[intencion] = len(coincidencias)

  mejor_intencion = max(puntajes, key=puntajes.get)
  mejor_puntaje = puntajes[mejor_intencion]

  if mejor_puntaje == 0:
    return "desconocida", 0

  return mejor_intencion, mejor_puntaje

In [56]:
detectar_intencion("a que hora abren el sabado?")


('consultar_horario', 5)

#Deteccion basica de entidades

Diccionario de palabras clave por servicio (busqueda de subcadenas en texto normalizado).

In [57]:
catalogo_servicios = {
    "internet": ["internet", "wifi", "red", "conexion"],
    "cuenta": ["cuenta", "usuario", "contrasena", "password","acceso"],
    "mantenimiento": ["mantenimiento", "reparacion", "servicio tecnico"],
    "facturacion": ["facturacion", "pago", "cobro", "recibo", "factura"]
}

def detectar_entidades(mensaje):
  """
  Detecta las entidades basicas mediante diccionario de terminos.
  """
  texto = normalizar_texto(mensaje)
  entidades = {}

  for entidad, palabras_clave in catalogo_servicios.items():
    for palabra in palabras_clave:
      if palabra in texto:
        entidades["servicio"] = entidad
        return entidades

  entidades["servicio"] = "no especificado"
  return entidades

In [58]:
detectar_entidades("no me llega el recibo de pago")

{'servicio': 'facturacion'}

#Generacion de Respuesta


In [59]:
from math import e
def generar_respuesta(mensaje):
  """
  Genera una respuesta a un mensaje. Considera intencion y entidades
  """
  intencion, puntaje = detectar_intencion(mensaje)
  entidades = detectar_entidades(mensaje)

  if intencion == "desconocida":
    return {
        "intencion": intencion,
        "puntaje": puntaje,
        "entidades": entidades,
        "respuesta": ("No estoy seguro de haber entendido, "
                      "Puedes pregunrarme por horarios, precios, soporte o servicios."
        )
    }

  plantilla = random.choice(intenciones[intencion]["respuestas"])
  respuestas = plantilla.format(**entidades)

  return {
      "intencion": intencion,
      "puntaje": puntaje,
      "entidades": entidades,
      "respuesta": respuestas
  }

In [63]:
for m in ["hola", "a que hora abren?", "cuanto cuesta el wifi",
          "mi factura tiene un cobro raro, necesito soporte", "trabajan los sabados", "adios"]:

          r = generar_respuesta(m)
          print(f"Usuario: {m}")
          print(f"  [{r['intencion']} | puntaje = {r['puntaje']} | {r['entidades']}]")
          print(f"Chatbot: {r['respuesta']}\n")

Usuario: hola
  [saludo | puntaje = 1 | {'servicio': 'no especificado'}]
Chatbot: Hola, con gusto te apoyo. Puedes preguntarme por horarios, precios o soporte tecnico.

Usuario: a que hora abren?
  [consultar_horario | puntaje = 4 | {'servicio': 'no especificado'}]
Chatbot: Nuestro horario de atencion es de lunes a viernes de 9:00 a 18:00 horas.

Usuario: cuanto cuesta el wifi
  [consultar_precio | puntaje = 3 | {'servicio': 'internet'}]
Chatbot: Los costos dependen del servicio solicitado. Detecte que te interesa: internet. ¿Deseas una cotizacion general?

Usuario: mi factura tiene un cobro raro, necesito soporte
  [solicitar_soporte | puntaje = 4 | {'servicio': 'facturacion'}]
Chatbot: Con gusto te apoyo. Para iniciar el reporte, necesito que me indiques mas detalles sobre el problema en facturacion.

Usuario: trabajan los sabados
  [consultar_horario | puntaje = 3 | {'servicio': 'no especificado'}]
Chatbot: Nuestro horario de atencion es de lunes a viernes de 9:00 a 18:00 horas.

Us

#Prueba interactiva del chatbot

Ejecuta la celda y escirbe tus mensajes. Escribe salir para terminar; el historial de la convesracion se regresa como DataFrame

In [61]:
def iniciar_chatbot():
  """
  Ejecuta el chatbot en modo interactivo dentro de Google Colab.
  """

  historial = []

  print("Chatbot: Hola, soy tu asistente virutal de servicio a cliente.")
  print("Chatbot: Puedes preguntar por horarios, precios, soporte o servicios.")
  print("Chatbot: Escribe salir para terminar el chat.\n")

  while True:
    mensaje_usuario = input("Usuario: ")

    if normalizar_texto(mensaje_usuario) == 'salir':
      print("Chatbot: Hasta luego. Que tengas un buen dia.")
      break

    resultado = generar_respuesta(mensaje_usuario)

    historial.append({
        "mensaje_usuario": mensaje_usuario,
        "intencion_detectada": resultado["intencion"],
        "puntaje_intencion": resultado["puntaje"],
        "entidades_detectadas": resultado["entidades"],
        "respuesta_chatbot": resultado["respuesta"]
    })

  return pd.DataFrame(historial)


In [62]:
#Ejecutar Chatbot

historial_chat = iniciar_chatbot()
historial_chat

Chatbot: Hola, soy tu asistente virutal de servicio a cliente.
Chatbot: Puedes preguntar por horarios, precios, soporte o servicios.
Chatbot: Escribe salir para terminar el chat.

Usuario: hola
Usuario: que onda
Usuario: 
Usuario: salir
Chatbot: Hasta luego. Que tengas un buen dia.


,mensaje_usuario,intencion_detectada,puntaje_intencion,entidades_detectadas,respuesta_chatbot
0,hola,saludo,1,{'servicio': 'no especificado'},"Hola, con gusto te apoyo. Puedes preguntarme p..."
1,que onda,saludo,1,{'servicio': 'no especificado'},"Hola, soy tu asistente virtual. Puedo ayudarte..."
2,,desconocida,0,{'servicio': 'no especificado'},"No estoy seguro de haber entendido, Puedes pre..."


#Evaluacion basica del chatbot

*   Precision de intencion: proporcion de casos con la intencion esperada.
*   Tasa de fallback: proporcion de mensajes clasificados como desconocida.



In [66]:
casos_prueba = [
    {"mensaje": "hola buenos dias", "intencion_esperada": "saludo"},
    {"mensaje": "cual es su horario de atencion", "intencion_esperada": "consultar_horario"},
    {"mensaje": "quiero saber el precio de internet", "intencion_esperada": "consultar_precio"},
    {"mensaje": "tengo un problema con mi cuenta", "intencion_esperada": "solicitar_soporte"},
    {"mensaje": "gracias hasta luego", "intencion_esperada": "despedida"},
    {"mensaje": "quiero cambiar mi correo electronico", "intencion_esperada": "desconocida"}
]

resultados = []

for caso in casos_prueba:
    resultado = generar_respuesta(caso["mensaje"])
    resultados.append({
        "mensaje": caso["mensaje"],
        "intencion_esperada": caso["intencion_esperada"],
        "intencion_detectada": resultado["intencion"],
        "correcto": caso["intencion_esperada"] == resultado["intencion"],
        "respuesta": resultado["respuesta"]
    })

df_resultados = pd.DataFrame(resultados)

precision = df_resultados["correcto"].mean()
tasa_fallback = (df_resultados["intencion_detectada"] == "desconocida").mean()

print("Resultados de evaluación")
print(f"  Precision de intencion: {precision:.2f}")
print(f"  Tasa de fallback: {tasa_fallback:.2f}")

df_resultados

Resultados de evaluación
  Precision de intencion: 0.83
  Tasa de fallback: 0.00


,mensaje,intencion_esperada,intencion_detectada,correcto,respuesta
0,hola buenos dias,saludo,saludo,True,"Hola, con gusto te apoyo. Puedes preguntarme p..."
1,cual es su horario de atencion,consultar_horario,consultar_horario,True,Nuestro horario de atencion es de lunes a vier...
2,quiero saber el precio de internet,consultar_precio,consultar_precio,True,Puedo orientarte con los precios. El servicio ...
3,tengo un problema con mi cuenta,solicitar_soporte,solicitar_soporte,True,Lamento el inconveniente. Detecte el servicio:...
4,gracias hasta luego,despedida,despedida,True,De nada. Que tengas un buen dia.
5,quiero cambiar mi correo electronico,desconocida,solicitar_soporte,False,"Con gusto te apoyo. Para iniciar el reporte, n..."
